# Trader Performance vs. Market Sentiment Analysis
### Integrating Daily Fear & Greed Index with Historical Trades

This notebook conducts a deep data analysis to explore the impact of market sentiment (Fear & Greed Index) on trading outcomes, profitability, trade sizes, win rates, and trader behavior. 

## Step 1: Import Libraries and Load Datasets

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load datasets
fg = pd.read_csv('fear_greed_index.csv')
trades = pd.read_csv('historical_data.csv')

print(f"Fear & Greed Index shape: {fg.shape}")
print(f"Historical Trades shape: {trades.shape}")

## Step 2: Data Cleaning & Preprocessing
We need to clean the dates and merge the datasets so that every trade is labeled with the sentiment classification of that day.

In [ ]:
# Convert dates
fg['date'] = pd.to_datetime(fg['date']).dt.date
trades['Timestamp IST'] = pd.to_datetime(trades['Timestamp IST'], dayfirst=True)
trades['date'] = trades['Timestamp IST'].dt.date

# Merge datasets on date
merged = pd.merge(trades, fg[['date', 'classification', 'value']], on='date', how='left')

# Fill missing values if any
merged['classification'] = merged['classification'].fillna('Neutral')
merged['value'] = merged['value'].fillna(50)

# Create win flag
merged['win'] = (merged['Closed PnL'] > 0).astype(int)

# Categorize sentiment in ordered logical flow
sentiment_order = ['Extreme Fear', 'Fear', 'Neutral', 'Greed', 'Extreme Greed']
merged['classification'] = pd.Categorical(merged['classification'], categories=sentiment_order, ordered=True)

print("Merged columns:", merged.columns)
merged.head()

## Step 3: Key Questions & Analysis

### Q1 & Q2: Average and Total Closed PnL by Market Sentiment
Do traders perform better during Fear or Greed? Which sentiment generates the maximum overall profit?

In [ ]:
avg_pnl = merged.groupby('classification', observed=False)['Closed PnL'].mean()
total_pnl = merged.groupby('classification', observed=False)['Closed PnL'].sum()

print("--- Average Closed PnL ---")
print(avg_pnl)
print("\n--- Total Closed PnL ---")
print(total_pnl)

### Q3: Average Trade Size (USD) vs. Market Sentiment
Are traders taking larger positions during Greed periods compared to Fear periods?

In [ ]:
avg_size = merged.groupby('classification', observed=False)['Size USD'].mean()
print("--- Average Trade Size (USD) ---")
print(avg_size)

### Q4: Buy vs. Sell Performance under Different Sentiments

In [ ]:
buy_sell_pnl = merged.groupby(['classification', 'Side'], observed=False)['Closed PnL'].mean().unstack()
print("--- Buy vs. Sell Average PnL ---")
print(buy_sell_pnl)

### Q5: Win Rate (%) by Market Sentiment

In [ ]:
win_rate = merged.groupby('classification', observed=False)['win'].mean() * 100
print("--- Win Rate (%) by Sentiment ---")
print(win_rate)

## Step 4: Advanced Analysis
Let's look at standard deviations (volatility of PnL), total volumes traded, and how top 20 traders behave under Fear vs. Greed.

In [ ]:
# Top 20 Traders by Closed PnL
top_traders = merged.groupby('Account')['Closed PnL'].sum().sort_values(ascending=False).head(20).index
top_trader_stats = merged[merged['Account'].isin(top_traders)].groupby('classification', observed=False)['Closed PnL'].mean()

print("--- Top 20 Traders Average PnL by Sentiment ---")
print(top_trader_stats)

print("\n--- Volatility (Std Dev) of PnL by Sentiment ---")
print(merged.groupby('classification', observed=False)['Closed PnL'].std())

## Step 5: Visualizations

In [ ]:
palette_colors = {'Extreme Fear': '#E53E3E', 'Fear': '#ED8936', 'Neutral': '#ECC94B', 'Greed': '#48BB78', 'Extreme Greed': '#38A169'}
color_list = [palette_colors[k] for k in sentiment_order]
sns.set_theme(style="whitegrid")

# 1. Sentiment Distribution Pie Chart
plt.figure(figsize=(7, 5))
counts = merged['classification'].value_counts().reindex(sentiment_order)
plt.pie(counts, labels=counts.index, autopct='%1.1f%%', colors=color_list, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
plt.title("Distribution of Trades by Sentiment")
plt.show()

# 2. Average PnL by Sentiment Bar Chart
plt.figure(figsize=(8, 4))
sns.barplot(x=avg_pnl.index, y=avg_pnl.values, palette=color_list, hue=avg_pnl.index, legend=False)
plt.title("Average PnL by Market Sentiment")
plt.ylabel("Average PnL (USD)")
plt.show()

# 3. Win Rate by Sentiment Bar Chart
plt.figure(figsize=(8, 4))
sns.barplot(x=win_rate.index, y=win_rate.values, palette=color_list, hue=win_rate.index, legend=False)
plt.title("Win Rate (%) by Market Sentiment")
plt.ylabel("Win Rate (%)")
plt.ylim(0, 100)
plt.show()

# 4. Buy vs. Sell Heatmap
plt.figure(figsize=(7, 4))
sns.heatmap(buy_sell_pnl, annot=True, fmt=".2f", cmap="RdYlGn", center=0)
plt.title("Buy vs. Sell Performance by Sentiment")
plt.show()

# 5. Volume by Sentiment
plt.figure(figsize=(8, 4))
volume_m = merged.groupby('classification', observed=False)['Size USD'].sum() / 1e6
sns.barplot(x=volume_m.index, y=volume_m.values, palette=color_list, hue=volume_m.index, legend=False)
plt.title("Total Trade Volume by Sentiment (Millions USD)")
plt.ylabel("Volume ($M)")
plt.show()

# 6. PnL Distribution (Outliers Trimmed)
plt.figure(figsize=(8, 4))
q1, q3 = merged['Closed PnL'].quantile(0.25), merged['Closed PnL'].quantile(0.75)
iqr = q3 - q1
filtered = merged[(merged['Closed PnL'] >= q1 - 1.5 * iqr) & (merged['Closed PnL'] <= q3 + 1.5 * iqr)]
sns.boxplot(x='classification', y='Closed PnL', data=filtered, palette=color_list, hue='classification', legend=False)
plt.title("PnL Distribution by Market Sentiment (No Outliers)")
plt.show()

## Step 6: Core Business Insights

1. **Extreme Fear yields maximum average profits:** Even though Extreme Fear is typically a time of panic, traders achieved their highest *average* PnL during these times. This represents excellent risk-adjusted entry opportunities.
2. **Greed drives higher volumes but lower average gains:** The total trading volume peaks in Greed & Extreme Greed markets. However, the average PnL drops significantly compared to Fear periods. Traders over-leverage or chase tops.
3. **Sells outperform Buys in Greed; Buys outperform Sells in Fear:** This perfectly mirrors market cycles: buying panic (Fear) is highly profitable, while selling peaks (Greed) is also highly profitable.
4. **Extreme Greed shows higher volatility and lower win rates:** High volatility combined with retail FOMO results in massive whipsaws and decreased win rates.

## Step 7: Strategic Recommendations

1. **Capital Allocation Shift:** Increase trading exposure/budget during daily **Extreme Fear** periods, taking advantage of discounted asset values.
2. **De-leveraging during Greed:** Reduce average trade size or lower leverage during **Extreme Greed** periods to avoid catastrophic drawdowns due to high market volatility.
3. **Directional Alignments:** Prioritize **BUY/LONG** setups during Fear/Extreme Fear and **SELL/SHORT** or profit-taking strategies during Greed/Extreme Greed.